# Setting Up the IRIS ML Pipeline on Vertex AI


## Overview

Build an end-to-end machine learning pipeline for the IRIS classifier on Google Cloud's Vertex AI platform, using Google Cloud Storage for data and artifact management.

### Objectives

* Set up and navigate the Google Cloud Platform and Vertex AI Workbench.
* Use Google Cloud Storage for ML data and artifact management.
* Build and execute an end-to-end IRIS classification pipeline on cloud infrastructure.
* Organize model artifacts by execution timestamp for traceability.
* Separate training and inference into distinct, reproducible scripts.

## Task 1

* Setup pipeline in git repository (https://github.com/21f1006125-ds/21f1006125_MLOPS_WEEKLY_ASSIGNMENT)
* Added IITMBSMLOps (da5014_1@study.iitm.ac.in) as collaborator

## Task 2
* Activated my GCP trial account and set up a Vertex AI Workbench instance. 
* Enabled all appropriate services and APIs as required.
    * Vertex AI
    * Cloud Storage
    * Compute Engine
* Provided the following permissions to service account (434534994925-compute@developer.gserviceaccount.com)
    * Cloud Run Builder
    * Storage Admin (Bucket creation)
    * Storage Object User (CRUD access for bucket objects)

## Task 3
* Created a Google Cloud Storage bucket and uploaded the IRIS dataset to it.
* Split the data into training and evaluation sets as you deem fit.

### Set Google Cloud project information

In [25]:
PROJECT_ID = "project-eada5958-ab21-4f76-b53"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Set up GCS Bucket
URL - https://console.cloud.google.com/storage/browser/mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments

In [26]:
BUCKET_URI = f"gs://mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [29]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Creating gs://mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments/...


### Configure Resource Names

In [55]:
DATA_DIR = "week-1/data"
MODEL_ARTIFACT_DIR = "week-1/models"  # @param {type:"string"}

### Upload data to GCS

In [41]:
! gcloud storage cp "Week 1/graded/data/iris.csv" {BUCKET_URI}/{DATA_DIR}/

Copying file://Week 1/graded/data/iris.csv to gs://mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments/week-1/data/iris.csv
  Completed files 1/1 | 3.8kiB/3.8kiB                                          


### Install dependencies

In [31]:
import sys
print(sys.executable)

/opt/micromamba/envs/jupyterlab/bin/python3


In [32]:
!{sys.executable} -m pip install -U google-cloud-aiplatform

In [33]:
!{sys.executable} -m pip show google-cloud-aiplatform

Name: google-cloud-aiplatform
Version: 1.158.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages
Requires: certifi, docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cloud-resource-manager, google-cloud-storage, google-genai, packaging, proto-plus, protobuf, pydantic, typing_extensions
Required-by: 


### Initialize Vertex AI SDK for Python


In [34]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [35]:
import os
import sys

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [36]:
import sys

!{sys.executable} -m pip install scikit-learn

In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

data = pd.read_csv('Week 1/graded/data/iris.csv')
data.head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [43]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

## Task 4
* Executed the IRIS machine learning training pipeline.
* Stored output artifacts (models, logs, etc.) in a GCS bucket with folders organized by their training execution timestamp.

In [50]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


In [60]:
import joblib
from datetime import datetime

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

artifact_dir = f"Week 1/graded/artifacts"
os.makedirs(artifact_dir, exist_ok=True)

joblib.dump(mod_dt, f"{artifact_dir}/model.joblib")

['Week 1/graded/artifacts/model.joblib']

### Upload model artifacts and custom code to Cloud Storage

Before you can deploy your model for serving, Vertex AI needs access to the following files in Cloud Storage:

* `model.joblib` (model artifact)
* `preprocessor.pkl` (model artifact)

Run the following commands to upload your files:

In [61]:
!gcloud storage cp "{artifact_dir}/model.joblib" {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{timestamp}/

Copying file://Week 1/graded/artifacts/model.joblib to gs://mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments/week-1/models/20260621_171939/model.joblib
  Completed files 1/1 | 2.5kiB/2.5kiB                                          


## Task 5
* Created a separate inference script that fetches the trained model from the GCS output artifacts bucket
* Ran inference on the evaluation set.

In [64]:
! gcloud storage cp {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{timestamp}/model.joblib "{artifact_dir}/model_inference.joblib"

Copying gs://mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments/week-1/models/20260621_171939/model.joblib to file://Week 1/graded/artifacts/model_inference.joblib
  Completed files 1/1 | 2.5kiB/2.5kiB                                          


In [67]:
model_inference = joblib.load(f"{artifact_dir}/model_inference.joblib")

predictions_inference = model_inference.predict(X_test)
print('The accuracy of the Inference Model (Decision Tree) is',"{:.3f}".format(metrics.accuracy_score(predictions_inference,y_test)))

The accuracy of the Inference Model (Decision Tree) is 0.983


## Task 6
* Execute the training and inference pipeline two times, resulting in two separate output artifact folders in your GCS bucket (each organized by execution timestamp).

In [69]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 20)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [70]:
mod_dt = DecisionTreeClassifier(max_depth = 4, random_state = 20)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.967


In [71]:
import joblib
from datetime import datetime

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

artifact_dir = f"Week 1/graded/artifacts"
os.makedirs(artifact_dir, exist_ok=True)

joblib.dump(mod_dt, f"{artifact_dir}/model_2.joblib")
!gcloud storage cp "{artifact_dir}/model_2.joblib" {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{timestamp}/
! gcloud storage cp {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{timestamp}/model_2.joblib "{artifact_dir}/model_2_inference.joblib"
model_inference_2 = joblib.load(f"{artifact_dir}/model_2_inference.joblib")

predictions_2_inference = model_inference_2.predict(X_test)
print('The accuracy of the Intereference model 2 is',"{:.3f}".format(metrics.accuracy_score(predictions_2_inference,y_test)))

Copying file://Week 1/graded/artifacts/model_2.joblib to gs://mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments/week-1/models/20260621_173321/model_2.joblib
  Completed files 1/1 | 2.5kiB/2.5kiB                                          
Copying gs://mlops-course-project-eada5958-ab21-4f76-b53-graded-assignments/week-1/models/20260621_173321/model_2.joblib to file://Week 1/graded/artifacts/model_2_inference.joblib
  Completed files 1/1 | 2.5kiB/2.5kiB                                          
The accuracy of the Intereference model 2 is 0.967
